# NB05_fPINN — How much of PINNmem's "memory" is really memory?

**Purpose.** In NB04 the integer baseline `PINNmem` adds a free linear memory Σ_j w_j·D^{α_j} q over the 39 nodes of `ALPHA_GRID` (0.05 … 1.0). Its memory branch carried 45 ± 9 mN·m RMS, more than the true fractional torque (11.0 mN·m). The last node is **α = 1**, where the L1 feature would be the backward difference of q, i.e. **q̇**; because the rigid-body PINN has no q̇ term, such a node could absorb the **viscous friction** b_v·q̇ of τ_u (b_v = 0.05), which is not memory.

**What the code actually does at α = 1.** In `caputo_l1` the L1 weights are b_k = (k+1)^{1−α} − k^{1−α}; at α = 1 every b_k is 1 − 1 = 0 (NumPy takes 0⁰ = 1), so the α = 1 feature is **identically zero** and its gain never trains. The nodes just below it are the ones that look like q̇: at α = 0.975 the kernel is b_0 = 1 followed by a tail below 0.018. This notebook checks both points and measures how much of the PINNmem branch is viscous friction, other τ_u terms, or fractional memory.

This notebook **retrains nothing**. It loads the saved NB04 weights (`Wm`, and `α̂`, `B̂` of the fPINN for reference), rebuilds the NB04 **test set** exactly (same seed; the 175 + 20 earlier trajectories are only drawn to advance the random stream), and splits the memory torque of every network into:

- all 39 nodes (the NB04 number), nodes with α < 0.975, the near-q̇ node α = 0.975, nodes with α < 0.9 and nodes with α ≥ 0.9, and the α = 1 node alone (expected 0);
- a per-joint least-squares projection of each branch onto the three true torques it could be absorbing: the fractional torque B·D^{0.5}q, the viscous torque b_v·q̇, and the rest of τ_u (Coulomb, Coriolis-like and sin 3q terms), with the residual.

A built-in self-check uses a synthetic `Wm` (B at α = 0.5, b_v at α = 0.975) whose gains must come out ≈ (1, 1, 0).

**One switch.** `QUICK = True`: 3 test trajectories and at most 2 networks (seconds). `QUICK = False`: the full test set (20 trajectories) and every PINNmem and fPINN network found in `NB04_fPINN_weights`.

**Expected runtime (Ryzen 7 PRO 5850U): about 1 min** (QUICK ≈ 15 s). RAM < 1 GB.

**Put in the same folder:** this notebook and the folder `NB04_fPINN_weights` of NB04 (only the `*_PINNmem_*` and `*_fPINN_*` files are read).

**Output:** `NB05_fPINN_MemoriaPINNmem.csv`.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/Anaconda: same fix as NB04
import glob, re, time, numpy as np, pandas as pd
from math import gamma as Gamma
from scipy.signal import fftconvolve

# ---- settings ---------------------------------------------------------------------------------
QUICK       = True                 # True: fast check (seconds). False: full test set, every network.
WEIGHTS_DIR = "NB04_fPINN_weights"
OUT_CSV     = "NB05_fPINN_MemoriaPINNmem" + ("_QUICK" if QUICK else "") + ".csv"
# ---- identical to NB04 ------------------------------------------------------------------------
B_FRAC, ALPHA_MAIN = 0.05, 0.5
FS = 2000; dt = 1.0/FS; SUBSAMPLE = 5
DATA_SEED, N_TRAIN_MAX, N_VAL_TRAJ, N_TEST_TRAJ = 12345, 175, 20, 20
ALPHA_GRID = np.round(np.arange(0.05, 1.0001, 0.025), 3); NA = len(ALPHA_GRID)
N_TEST_USE = 3 if QUICK else N_TEST_TRAJ
print("QUICK" if QUICK else "FULL", f"| α grid {ALPHA_GRID[0]}…{ALPHA_GRID[-1]} ({NA} nodes)")

## Robot model and trajectories (copied unchanged from NB04)

In [ ]:
R_BASE, R_PLAT, L_A, L_B = 0.114, 0.035, 0.200, 0.460
THETA0 = np.array([0.070, 0.023, 1e-5, 0.006, 0.253, 1.12, 0.008, 0.006, 0.006, 1.0, 0.008, 0.006, 0.006, 1.0])
M_BASE = 6.0
UNMODELED = dict(b_v=0.05, f_c=0.04, eps=0.02, c_cor=0.06, k_pos=0.04, w_pos=3.0)
def gamma_payload(m): return (M_BASE + np.asarray(m, float)) / M_BASE

def ik_vec(P):
    x, y, z = P[:, 0], P[:, 1], P[:, 2]; q = np.zeros((len(P), 3)); rd = R_BASE - R_PLAT
    for i in range(3):
        al = 2*i*np.pi/3; u = -2*L_A*z; aux = x*np.cos(al) + y*np.sin(al)
        v = 2*L_A*(rd - aux); w = L_B**2 - (rd**2 - 2*rd*aux + x**2 + y**2 + z**2 + L_A**2)
        r = np.sqrt(u**2 + v**2 - w**2); den = u**2 + v**2
        s1 = (u*w + v*r)/den; c1 = (v*w - u*r)/den; t1 = np.where(c1 >= 0, np.arcsin(s1), np.pi - np.arcsin(s1))
        s2 = (u*w - v*r)/den; c2 = (v*w + u*r)/den; t2 = np.where(c2 >= 0, np.arcsin(s2), np.pi - np.arcsin(s2))
        q[:, i] = np.where(np.abs(t1) < np.abs(t2), t1, t2)
    return q

def fk_vec(q):
    """Forward kinematics of NB03 (fk), vectorized over rows of q (n, 3)."""
    rd = R_BASE - R_PLAT; C = np.zeros((len(q), 3, 3))
    for i in range(3):
        a = 2*i*np.pi/3; r = rd + L_A*np.cos(q[:, i])
        C[:, i] = np.column_stack([r*np.cos(a), r*np.sin(a), L_A*np.sin(q[:, i])])
    A1, B1, C1 = (2*(C[:, 1] - C[:, 0])).T; D1 = 2*L_A*rd*(np.cos(q[:, 0]) - np.cos(q[:, 1]))
    A2, B2, C2 = (2*(C[:, 2] - C[:, 0])).T; D2 = 2*L_A*rd*(np.cos(q[:, 0]) - np.cos(q[:, 2]))
    den = A2*B1 - A1*B2
    L1 = (A1*D2 - A2*D1)/den; L2 = (A1*C2 - A2*C1)/den; L3 = (B2*D1 - B1*D2)/den; L4 = (B2*C1 - B1*C2)/den
    u = 1 + L2**2 + L4**2; c0 = C[:, 0]
    v = 2*(L3*L4 + L1*L2 - c0[:, 0]*L4 - c0[:, 1]*L2 - c0[:, 2])
    w = c0[:, 0]**2 + c0[:, 1]**2 + c0[:, 2]**2 + L3**2 + L1**2 - 2*c0[:, 0]*L3 - 2*c0[:, 1]*L1 - L_B**2
    z = (-v + np.sqrt(v**2 - 4*u*w)) / (2*u)
    return np.column_stack([L3 + L4*z, L1 + L2*z, z])
def fk(q): return fk_vec(q[None, :])[0]

def regressor_batch(q, a, P):
    n = len(q); s, c = np.sin(q), np.cos(q); x, y, z = P[:, 0], P[:, 1], P[:, 2]
    Y = np.zeros((n, 3, 14)); Y[:, :, 0] = a; Y[:, :, 4] = c
    for i, (cx, cy, cz, cs) in enumerate([(1, 2, 3, 5), (6, 7, 8, 9), (10, 11, 12, 13)]):
        Y[:, i, cx] = -x*s[:, i]; Y[:, i, cy] = -y*s[:, i]; Y[:, i, cz] = z*c[:, i]; Y[:, i, cs] = s[:, i]
    return Y
def regressor(q, a, p): return regressor_batch(q[None, :], np.asarray(a, float)[None, :], np.asarray(p)[None, :])[0]

def tau_unmodeled_batch(q, qd, sev=1.0, b_v=None):
    u = UNMODELED; bv = u["b_v"] if b_v is None else b_v
    return sev*(bv*qd + u["f_c"]*np.tanh(qd/u["eps"])
                + u["c_cor"]*(qd**2*np.cos(q) + np.roll(qd, -1, axis=1)*np.roll(qd, -2, axis=1))
                + u["k_pos"]*np.sin(u["w_pos"]*q))

def caputo_l1(x, h, alpha):
    x = np.asarray(x, float); N = len(x); k = np.arange(N); b = (k + 1.0)**(1 - alpha) - k**(1.0 - alpha)
    d = np.vstack([np.zeros((1, x.shape[1])), np.diff(x, axis=0)])
    return fftconvolve(d, b[:, None], axes=0)[:N] * h**(-alpha) / Gamma(2 - alpha)
def random_trajectory_full(rng, n_wp=6, v_range=(0.03, 0.8), t_min=0.3):     # RNG use identical to nb 18
    W = np.column_stack([rng.uniform(-0.12, 0.12, n_wp), rng.uniform(-0.12, 0.12, n_wp), rng.uniform(0.49, 0.56, n_wp)])
    dist = np.linalg.norm(np.diff(W, axis=0), axis=1)
    v = np.exp(rng.uniform(np.log(v_range[0]), np.log(v_range[1]), n_wp - 1))
    dur = np.maximum(dist / v, t_min); m_load = float(rng.integers(0, 7))
    edges = np.concatenate([[0.0], np.cumsum(dur)]); t_g = np.arange(0.0, edges[-1], dt)
    seg = np.clip(np.searchsorted(edges, t_g, side="right") - 1, 0, n_wp - 2)
    ts = (t_g - edges[seg]) / dur[seg]; s = 6*ts**5 - 15*ts**4 + 10*ts**3
    P = W[seg] + (W[seg + 1] - W[seg])*s[:, None]
    q = ik_vec(P); qd = np.gradient(q, dt, axis=0, edge_order=2); qdd = np.gradient(qd, dt, axis=0, edge_order=2)
    return dict(q=q, qd=qd, qdd=qdd, P=P, m=m_load)

## NB04 test set: features D^{α_j} q at 2.5 ms, true fractional and viscous torques

In [ ]:
def log_test(T, margin_s=0.05):
    """Same sampling as log_and_label of NB04 (clean data)."""
    q, qd = T["q"], T["qd"]
    mg = int(margin_s/dt); idx = np.arange(0, len(q), SUBSAMPLE); keep = (idx >= mg) & (idx < len(q) - mg)
    h = dt*SUBSAMPLE; ql = q[idx]; i = idx[keep]
    G = np.stack([caputo_l1(ql, h, a)[keep] for a in ALPHA_GRID], axis=1)              # (n, NA, 3)
    tu = tau_unmodeled_batch(q, qd)[i]; visc = UNMODELED["b_v"]*qd[i]
    return dict(G=G, frac=B_FRAC*caputo_l1(q, dt, ALPHA_MAIN)[i], visc=visc, other=tu - visc)

t0 = time.time(); rng = np.random.default_rng(DATA_SEED)
for _ in range(N_TRAIN_MAX + N_VAL_TRAJ): random_trajectory_full(rng)                 # advance the stream: train + val
parts = [log_test(random_trajectory_full(rng)) for _ in range(N_TEST_TRAJ)][:N_TEST_USE]
TE = {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}
rms = lambda x: float(np.sqrt(np.mean(np.asarray(x)**2)))
print(f"test set: {N_TEST_USE} trajectories, {len(TE['G']):,} samples in {time.time()-t0:.0f} s | "
      f"true fractional torque {1e3*rms(TE['frac']):.2f} mN m | true viscous b_v q̇ {1e3*rms(TE['visc']):.2f} | other τ_u {1e3*rms(TE['other']):.2f} mN m")
print(f"α = 1 feature: max |D^1 q| = {np.abs(TE['G'][:, -1]).max():.3g} (0 means the node is empty) | "
      f"α = 0.975 feature vs q̇: correlation {np.corrcoef(TE['G'][:, -2].ravel(), TE['visc'].ravel())[0, 1]:.4f}")

## Decomposition of a memory branch

In [ ]:
I_A1 = int(np.argmin(abs(ALPHA_GRID - 1.0))); I_0975 = int(np.argmin(abs(ALPHA_GRID - 0.975)))
SPLITS = {"all": np.ones(NA, bool), "a<0.975": ALPHA_GRID < 0.975 - 1e-9, "a=0.975": np.arange(NA) == I_0975,
          "a<0.9": ALPHA_GRID < 0.9 - 1e-9, "a>=0.9": ALPHA_GRID > 0.9 - 1e-9, "a=1": np.arange(NA) == I_A1}

def interp_D(G, alpha):                       # numpy copy of NB04 interp_D
    u = np.clip((alpha - ALPHA_GRID[0])/(ALPHA_GRID[1] - ALPHA_GRID[0]), 0.0, NA - 1.000001)
    i0 = int(np.floor(u)); w = u - i0
    return (1 - w)*G[:, i0, :] + w*G[:, i0 + 1, :]

def project(m, d):
    """Per joint LS fit m ≈ c_f·frac + c_v·visc + c_o·other; returns gains averaged over joints and the residual RMS."""
    C, res = [], []
    for j in range(3):
        A = np.column_stack([d["frac"][:, j], d["visc"][:, j], d["other"][:, j]]); c, *_ = np.linalg.lstsq(A, m[:, j], rcond=None)
        C.append(c); res.append(m[:, j] - A @ c)
    return (*np.mean(C, 0).astype(float), rms(np.column_stack(res)))

def decompose(Wm, d):
    G = d["G"]; parts = {k: (G[:, s]*Wm[s]).sum(1) for k, s in SPLITS.items()}
    out = {f"Memory RMS {k} (mN m)": 1e3*rms(v) for k, v in parts.items()}
    for k in ("all", "a<0.9", "a>=0.9"):
        cf, cv, co, r = project(parts[k], d)
        out.update({f"Gain frac {k}": cf, f"Gain visc {k}": cv, f"Gain other {k}": co, f"Residual {k} (mN m)": 1e3*r,
                    f"Error vs true frac {k} (mN m)": 1e3*rms(parts[k] - d["frac"])})
    out["Wm a=0.975"] = [round(float(w), 4) for w in Wm[I_0975]]; out["Wm a=1"] = [round(float(w), 4) for w in Wm[I_A1]]
    return out

# self-check: B at α = 0.5 and b_v at α = 0.975 must give gains near (frac 1, visc 1, other 0); not exact because D^0.975 q is not exactly q̇
W_chk = np.zeros((NA, 3)); W_chk[int(np.argmin(abs(ALPHA_GRID - 0.5)))] = B_FRAC; W_chk[I_0975] = UNMODELED["b_v"]
chk = decompose(W_chk, TE)
print("self-check | memory α<0.9 %.2f mN m (true fractional %.2f) | α=0.975 %.2f (true viscous %.2f) | gains frac %.3f visc %.3f other %.3f" % (
      chk["Memory RMS a<0.9 (mN m)"], 1e3*rms(TE["frac"]), chk["Memory RMS a=0.975 (mN m)"], 1e3*rms(TE["visc"]),
      chk["Gain frac all"], chk["Gain visc all"], chk["Gain other all"]))

## Every PINNmem network of NB04 (and every fPINN network, for reference)

In [ ]:
files = sorted(f for f in glob.glob(os.path.join(WEIGHTS_DIR, "NB04_fPINN_*.npz"))
               if re.search(r"_(hist|main)_(PINNmem|fPINN)_\d+traj_seed\d+_a0\.5_e150\.npz$", os.path.basename(f)))   # e150 only: QUICK networks of NB04 end in _e2
if QUICK: files = [f for f in files if "_PINNmem_" in f][:1] + [f for f in files if "_fPINN_" in f][:1]
if not files: print(f"no NB04 weights found in '{WEIGHTS_DIR}': put that folder next to this notebook (only the self-check ran)")
rows = []
for f in files:
    mt = re.search(r"_(hist|main)_(PINNmem|fPINN)_(\d+)traj_seed(\d+)_", os.path.basename(f)); W = np.load(f)
    assert np.allclose(W["alpha_grid"], ALPHA_GRID)
    if mt.group(2) == "PINNmem": Wm = W["Wm"]
    else:                                                    # fPINN: B̂·D^α̂ q written as an equivalent Wm on the grid
        Wm = np.zeros((NA, 3)); a = float(W["alpha"]); u = min(max((a - ALPHA_GRID[0])/(ALPHA_GRID[1] - ALPHA_GRID[0]), 0), NA - 1.000001)
        i0 = int(np.floor(u)); Wm[i0] = (1 - (u - i0))*W["B"]; Wm[i0 + 1] = (u - i0)*W["B"]
    row = dict(Kind=mt.group(2), Scenario=mt.group(1), Trajectories=int(mt.group(3)), Seed=int(mt.group(4)),
               **{"alpha hat": float(W["alpha"]) if mt.group(2) == "fPINN" else np.nan}, **decompose(Wm, TE))
    rows.append(row)
    print(f"{row['Kind']:7s} {row['Trajectories']:3d} traj seed {row['Seed']} | memory all {row['Memory RMS all (mN m)']:6.2f} | "
          f"α<0.9 {row['Memory RMS a<0.9 (mN m)']:6.2f} | α≥0.9 {row['Memory RMS a>=0.9 (mN m)']:6.2f} | α=1 {row['Memory RMS a=1 (mN m)']:.2f} mN m | "
          f"gains (frac, visc, other) {row['Gain frac all']:.2f}, {row['Gain visc all']:.2f}, {row['Gain other all']:.2f} | resid {row['Residual all (mN m)']:.2f}")
df = pd.DataFrame(rows + [dict(Kind="self-check", Scenario="-", Trajectories=0, Seed=-1, **chk)])
df.to_csv(OUT_CSV, index=False); print(f"wrote {OUT_CSV}")

## Summary (mean ± sd over seeds)

In [ ]:
if rows:
    cols = ([f"Memory RMS {k} (mN m)" for k in SPLITS] + [f"{g} {k}" for k in ("all", "a<0.9", "a>=0.9") for g in ("Gain frac", "Gain visc", "Gain other")]
            + ["Residual all (mN m)", "Error vs true frac all (mN m)"])
    S = pd.DataFrame(rows).groupby(["Kind", "Trajectories"])[cols].agg(lambda x: f"{x.mean():.3f} ± {x.std(ddof=1) if len(x) > 1 else 0:.3f}")
    with pd.option_context("display.width", 250, "display.max_columns", 20): print(S.T.to_string())
    print(f"\nreference: true fractional torque {1e3*rms(TE['frac']):.2f} mN m, true viscous torque {1e3*rms(TE['visc']):.2f} mN m, other τ_u {1e3*rms(TE['other']):.2f} mN m")